# Classifiers Playground
"How can I make a classification model that can classify four seconds of 8-channel EEG data into either movement or no movement?"

The suspiciously LDA shaped horse: 🙂‍↕️🙂‍↕️🙂‍↕️

In [ ]:
print("test")
import numpy as np
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

In [ ]:
# config
DATA_FOLDER = "data/"
DATA_SESSION = "3-4/joshfoot/"
SESSIONS = [7, 8, 9]
CHANNELS = [2, 3, 4, 6] # in theory these are the only ones that should matter

In [ ]:
# load filtered data
alltrials = np.empty((0, 2))
for session in SESSIONS:
    filtered_session = np.load(f"{DATA_FOLDER}{DATA_SESSION}filtered-session-{session}.npy", allow_pickle=True)
    for i in range(len(filtered_session)):
        filtered_session[i][1] = filtered_session[i][1][CHANNELS, 62:-100]
    alltrials = np.concatenate((alltrials, filtered_session), axis=0)

In [ ]:
# set up csp and lda labels
labels = np.array([1 if trial[0] == 'clench right foot' else 0 for trial in alltrials])
eeg = np.array([trial[1] for trial in alltrials])

print(labels.shape)
print(eeg.shape)

In [ ]:
# set up line that has a pipe
csp = CSP(n_components=2, reg='ledoit_wolf', log=True)
lda = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')

pl = Pipeline([
    ('csp', csp),
    ('lda', lda)
])

In [ ]:
# yall remember pa3 with the folding nonsense
# turns out there's straight up a built in library for that
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=67)
scores = cross_val_score(pl, eeg, labels, cv=cv, scoring='accuracy')
print(f"fold scores: {scores}")